# Lecture 21: Pandas: Working with Dates and Text

## Learning Objectives

- Parse dates with pd.to_datetime() and generate date ranges with pd.date_range()
- Extract date components using the .dt accessor
- Resample time series data with different aggregation functions
- Shift and difference time series with .shift() and .diff()
- Use the .str accessor for text cleaning and extraction
- Work with categorical data types for memory efficiency

## Key Topics

- pd.to_datetime() and pd.date_range()
- .dt accessor: .year, .month, .dayofweek, etc.
- resample() with aggregation
- Shifting and differencing: .shift(), .diff()
- .str accessor: .contains(), .extract(), .replace()
- Categorical data type and memory savings

### Working with Dates and Times

Date and time data requires special handling because of varying formats, time zones, and the need to extract components like year, month, or day of week. `pd.to_datetime()` converts strings, numbers, or Python datetime objects into Pandas Timestamp objects. `pd.date_range()` generates a fixed-frequency DatetimeIndex, which is useful for creating time series skeletons.

Once a column is in datetime format, the `.dt` accessor unlocks dozens of properties and methods. You can extract `.year`, `.month`, `.day`, `.dayofweek`, `.quarter`, and many more. This makes it trivial to create features like 'is_weekend' or 'monthly averages' from raw date columns.

In [ ]:
import pandas as pd
import numpy as np

# Parsing dates
dates = ['2024-01-15', '2024-02-20', '2024-03-25', 'invalid_date']
parsed = pd.to_datetime(dates, errors='coerce')
print('Parsed dates:', parsed.tolist())

# Generating date ranges
daily = pd.date_range(start='2024-01-01', end='2024-01-10', freq='D')
print('Daily:', daily.tolist())

business = pd.date_range(start='2024-01-01', periods=5, freq='B')
print('Business days:', business.tolist())

In [ ]:
# .dt accessor
df_dates = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=10, freq='D'),
    'value': np.random.randn(10)
})
df_dates['year'] = df_dates['date'].dt.year
df_dates['month'] = df_dates['date'].dt.month
df_dates['day'] = df_dates['date'].dt.day
df_dates['dayofweek'] = df_dates['date'].dt.dayofweek
df_dates['is_weekend'] = df_dates['date'].dt.dayofweek >= 5
print('Date components:')
print(df_dates)

### Resampling, Shifting, and Differencing

Time series data often needs to be aggregated to a different frequency. `resample()` is similar to `groupby()` but for time-based grouping. You specify a frequency string like `'M'` (month end), `'W'` (weekly), `'H'` (hourly), and an aggregation function. This is essential for rolling up high-frequency data (e.g., daily sales to monthly totals).

`shift()` moves data forward or backward in time, which is how you create lag features for forecasting models. `diff()` computes the difference between consecutive observations, which is useful for detrending a series or computing returns. Both are fundamental tools for time series feature engineering.

In [ ]:
# Resampling
idx = pd.date_range('2024-01-01', periods=90, freq='D')
ts = pd.Series(np.random.randn(90).cumsum(), index=idx, name='Price')

monthly = ts.resample('M').agg(['mean', 'min', 'max'])
print('Monthly resample:')
print(monthly.head())

In [ ]:
# Shifting and differencing
df_ts = pd.DataFrame({
    'value': [100, 102, 105, 103, 108, 110]
}, index=pd.date_range('2024-01-01', periods=6, freq='D'))

df_ts['lag_1'] = df_ts['value'].shift(1)
df_ts['diff'] = df_ts['value'].diff()
df_ts['pct_change'] = df_ts['value'].pct_change()
print('Shift and diff:')
print(df_ts)

### String Methods with .str Accessor

Pandas provides a `.str` accessor that gives you vectorised string operations, mimicking Python's string methods. `.str.contains()` checks if a pattern exists in each element. `.str.extract()` pulls out substrings matching a regex pattern. `.str.replace()` substitutes patterns. These methods handle missing values gracefully (returning NaN instead of crashing).

Text data in the real world is messy: inconsistent casing, extra whitespace, embedded symbols, and typos. The `.str` accessor lets you clean and normalise text columns efficiently across millions of rows, making it an indispensable tool for data preparation.

In [ ]:
# .str accessor for text cleaning
df_text = pd.DataFrame({
    'text': [
        'Order #1234 - PENDING',
        'Order #5678 - SHIPPED',
        'Order #9012 - delivered',
        None
    ]
})

df_text['lower'] = df_text['text'].str.lower()
df_text['has_shipped'] = df_text['text'].str.contains('shipped', case=False)
df_text['order_id'] = df_text['text'].str.extract(r'#(\d+)')
df_text['status'] = df_text['text'].str.extract(r'-\s*(\w+)')
df_text['status'] = df_text['status'].str.replace('pending', 'PENDING')
print('Text cleaning:')
print(df_text)

In [ ]:
# More string operations
reviews = pd.DataFrame({
    'review': [
        '  Great product!  ',
        'Not bad... but could be better',
        'Excellent! Highly recommended!'
    ]
})

reviews['clean'] = reviews['review'].str.strip()
reviews['word_count'] = reviews['clean'].str.split().str.len()
reviews['has_excellent'] = reviews['clean'].str.contains('excellent', case=False)
print('Review analysis:')
print(reviews)

### Categorical Data Type


Columns with a limited set of repeated values (like country codes, product categories, or yes/no) benefit from the `category` dtype. Instead of storing the full string for every row, Pandas stores a compact integer mapping internally. This can dramatically reduce memory usage, especially for columns with high cardinality.

The `category` dtype also enables categorical operations like `.value_counts()` with `dropna=False`, and you can specify an order with `pd.CategoricalDtype(categories=[...], ordered=True)`. Ordered categories are respected by `.sort_values()` and `.min()` / `.max()`, which is useful for ordinal data like education levels or survey responses.

In [ ]:
# Categorical data type and memory savings
n = 100_000
df_cat = pd.DataFrame({
    'category': np.random.choice(['Low', 'Medium', 'High', 'Critical'], n),
    'value': np.random.randn(n)
})

memory_obj = df_cat['category'].memory_usage(deep=True)
df_cat['category'] = df_cat['category'].astype('category')
memory_cat = df_cat['category'].memory_usage(deep=True)

print(f'Memory as object: {memory_obj / 1024:.1f} KB')
print(f'Memory as category: {memory_cat / 1024:.1f} KB')
print(f'Savings: {(1 - memory_cat / memory_obj) * 100:.1f}%')

In [ ]:
# Ordered categorical
df_order = pd.DataFrame({
    'education': ['BSc', 'PhD', 'MSc', 'BSc', 'PhD']
})

cat_type = pd.CategoricalDtype(
    categories=['BSc', 'MSc', 'PhD'], ordered=True
)
df_order['education'] = df_order['education'].astype(cat_type)

print('Sorted by education level:')
print(df_order.sort_values('education'))

## Data Science Connection

Date/time handling and text processing are two of the most common challenges in real-world data. Time series analysis powers forecasting, anomaly detection, and trend analysis across finance, IoT, and operations. Text processing is essential for cleaning survey data, parsing logs, and preparing natural language data. The categorical dtype is a simple optimisation that can dramatically reduce memory and speed up processing.